# Intro

- https://typesafe.ai/blog/introducing-system-one-models-and-jev
- https://github.com/jaredpalmer/kev
- https://github.com/jaredpalmer/kev/blob/main/docs/model-cards/kev-0.8b.md
- https://huggingface.co/jaredpalmer/kev-0.8b

# Part 1 - from the request to the probabilities

In [ ]:
import torch
from kev.checkpoint import Checkpoint, LoadOptions

ck = Checkpoint("jaredpalmer/kev-0.8b")      # downloads adapter, head.pt and config from the Hub into the HF cache
tok, model = ck.load("mps", LoadOptions())   # Qwen base + merged LoRA, pointer head loaded, eval mode

In [64]:
from kev.api import SystemOneRequest, to_record, to_answers

req = SystemOneRequest(
    state="Shoes arrived two weeks late and in the wrong size. Also I see two charges on my card.",
    questions={
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "returns": "Exchanges, refunds, wrong or damaged items",
                "shipping": "Delivery status, delays, lost packages",
                "billing": "Charges, invoices, payment problems",
            },
        }
    },
)

rec, meta = to_record(req)        # text: state + instructions + options "name: description"

In [65]:
enc = model.encode(tok, rec)      # token ids, positions, segments, indices of </opt> and <decide>
print(tok.decode(enc["ids"]))   # the whole sequence, special tokens included

<|fim_prefix|>Shoes arrived two weeks late and in the wrong size. Also I see two charges on my card.<|fim_middle|>Which team should handle this?<|box_start|>returns: Exchanges, refunds, wrong or damaged items<|box_end|><|box_start|>shipping: Delivery status, delays, lost packages<|box_end|><|box_start|>billing: Charges, invoices, payment problems<|box_end|><|fim_suffix|>


In [47]:
toks = [tok.decode([i]).replace("\n", "⏎") for i in enc["ids"]]
print(" | ".join(toks))

<|fim_prefix|> | Sh | oes |  arrived |  two |  weeks |  late |  and |  in |  the |  wrong |  size | . |  Also |  I |  see |  two |  charges |  on |  my |  card | . | <|fim_middle|> | Which |  team |  should |  handle |  this | ? | <|box_start|> | returns | : |  Ex | changes | , |  refunds | , |  wrong |  or |  damaged |  items | <|box_end|> | <|box_start|> | shipping | : |  Delivery |  status | , |  delays | , |  lost |  packages | <|box_end|> | <|box_start|> | billing | : |  Charges | , |  invoices | , |  payment |  problems | <|box_end|> | <|fim_suffix|>


In [11]:
probs= model.probs(enc)
probs

[tensor([0.2612, 0.4579, 0.2809])]

# Part 2 - the pointer head

In [15]:
enc["opt_idx"], enc["decide_idx"]

([[41, 52, 62]], [63])

In [16]:
ids = torch.tensor(enc["ids"],device="mps").reshape(1,-1)
ids.shape

torch.Size([1, 64])

In [18]:
with torch.no_grad():
    h = model.lm(input_ids=ids).last_hidden_state              # [1, L, hidden_size]
h.shape

torch.Size([1, 20, 1024])

In [33]:
h_decide = h[0, enc["decide_idx"][0]]   # [hidden_size], the <decide> token
h_opts   = h[0, enc["opt_idx"][0]]      # [3, hidden_size], the three </opt>: returns, shipping, billing

In [46]:
w_q = model.head.q              # Linear(hidden_size -> 256), projects h_decide
w_k = model.head.k              # Linear(hidden_size -> 256), projects the h of the options
scale = model.head.scale          # 1/16
temp = model.head.temperature    # checkpoint temperature

In [52]:
h_decide.shape, w_q 
q = w_q(h_decide)
q.shape

torch.Size([256])

In [53]:
h_opts.shape, w_k
k = w_k(h_opts)
k.shape

torch.Size([3, 256])

In [56]:
z = q@k.T * scale
z

tensor([2.4207, 3.7408, 2.5922], device='mps:0', grad_fn=<MulBackward0>)

In [57]:
p = torch.softmax(z / model.head.temperature, -1)   # [3], sums to 1

In [58]:
p

tensor([0.2612, 0.4579, 0.2809], device='mps:0', grad_fn=<SoftmaxBackward0>)

# Appendix - kev-suites dataset

In [8]:
from datasets import load_dataset

ds = load_dataset("jaredpalmer/kev-suites",
                  data_files={"development": "v4/transfer-v4/development.jsonl",
                              "test": "v4/transfer-v4/test.jsonl"})

dev  = ds["development"]   # 764 questions, to iterate on
test = ds["test"]          # 764 questions, only at the end

test.jsonl:   0%|          | 0.00/804k [00:00<?, ?B/s]

Generating development split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [13]:
dev

Dataset({
    features: ['state', 'questions', '_meta'],
    num_rows: 764
})

In [15]:
n = 1

In [18]:
dev["state"][n]

{'subject': 'clinical knowledge', 'question': 'What is hypochlorhydria?'}

In [19]:
dev["questions"][n]

{'answer': {'type': 'choice',
  'instructions': 'Which option correctly answers the question?',
  'criteria': {'a': 'Increased amount of gastric acid.',
   'b': 'Gastric ulceration.',
   'c': 'Reduced amount of gastric acid.',
   'd': 'Pernicious anaemia.'},
  'label': 'c',
  'src': 'mmlu'}}